# **Data Visualization for Thesis**

**Note on AI usage:** Visualizations were produced with assistance from ChatGPT. Visualization code was generated iteratively through user-guided prompts informed by the author’s own visualization principles and design choices.

### **Packages and Data**

In [1]:
import pandas as pd
import altair as alt

In [2]:
df = pd.read_csv("data/data.csv")

### **Parliamentary Entry Rates**

In [9]:
# Create main dataframe with all data for the visualization
df = pd.DataFrame({
    "group": [
        "Student leaders",
        "Female", "Male",
        "Röskva", "Vaka", "Unknown", "Other assn.",
        "University Council", "Student Council", "Student Council (1970s-)",
        "1930s–1950s", "1960s–1980s", "1990s–2020s",
        "1st position", "2nd position", "3rd position", "4th position", "5th position"
    ],
    # Success rates for each group (as decimal values, will be formatted as percentages)
    "rate": [
        0.0783,
        0.0981, 0.0694,
        0.0890, 0.0792, 0.0667, 0.0660,
        0.1447, 0.0721, 0.0612,
        0.1008, 0.0471, 0.0873,
        0.0823, 0.0741, 0.0667, 0.0742, 0.0297
    ],
    # Section categories for grouping data into separate charts
    "section": [
        "Student leader",
        "Gender", "Gender",
        "Association", "Association", "Association", "Association",
        "Council", "Council", "Council",
        "Cohort", "Cohort", "Cohort",
        "Ticket position", "Ticket position", "Ticket position", "Ticket position", "Ticket position"
    ],
    # Custom colors for each section (different shades of gray)
    "color": [
        "#3f3f3f",
        "#4f4f4f", "#4f4f4f",
        "#5f5f5f", "#5f5f5f", "#5f5f5f", "#5f5f5f",
        "#6f6f6f", "#6f6f6f", "#6f6f6f",
        "#7f7f7f", "#7f7f7f", "#7f7f7f",
        "#8f8f8f", "#8f8f8f", "#8f8f8f", "#8f8f8f", "#8f8f8f"
    ]
})

# Filter dataframe into separate datasets for each chart section
leaders = df[df["section"] == "Student leader"]
gender = df[df["section"] == "Gender"]
assn = df[df["section"] == "Association"]
councils = df[df["section"] == "Council"]
cohorts = df[df["section"] == "Cohort"]
ticket = df[df["section"] == "Ticket position"]

def make_chart(data, label, show_xaxis=False):
    # Get the order of groups from the data
    order = list(data["group"])
    # Store the first group for labeling purposes
    first_group = order[0]

    # Create horizontal bar chart with rounded corners
    bars = alt.Chart(data).mark_bar(
    size=18,
    cornerRadiusEnd=1
).encode(
    # Y-axis: groups in specified order, no title
    y=alt.Y("group:N", sort=order, title=None),
    # X-axis: rate values with fixed domain and conditional formatting
    x=alt.X(
        "rate:Q",
        scale=alt.Scale(domain=[0, 0.15]),
        title="" if show_xaxis else None,
        # Show full axis with percentage format if show_xaxis is True
        axis=alt.Axis(
            format="%",
            values=[0, 0.05, 0.10, 0.15],
            grid=False
        ) if show_xaxis else alt.Axis(
            # Hide axis elements if show_xaxis is False
            labels=False, ticks=False, domain=False, grid=False
        )
    ),
    # Use custom colors from data without legend
    color=alt.Color("color:N", scale=None, legend=None)
)

    # Add text labels showing percentage values on bars
    values = alt.Chart(data).mark_text(
        align="left",
        baseline="middle",
        dx=3  # Offset text slightly to the right
    ).encode(
        y=alt.Y("group:N", sort=order),
        x=alt.X("rate:Q", scale=alt.Scale(domain=[0, 0.15])),
        # Format rate as percentage with 1 decimal place
        text=alt.Text("rate:Q", format=".1%")
    )

    # Create dataframe for section label
    label_df = pd.DataFrame({
        "group": [first_group],
        "x0": [0],
        "label": [label]
    })

    # Add section label at the top of each chart
    section_label = alt.Chart(label_df).mark_text(
        align="left",
        baseline="bottom",
        dy=-9,  # Position label above the chart
        fontWeight="bold",
        fontSize=11
    ).encode(
        y=alt.Y("group:N", sort=order),
        x=alt.X("x0:Q", scale=alt.Scale(domain=[0, 0.15])),
        text="label:N"
    )

    # Combine all chart elements and set width
    return (section_label + bars + values).properties(width=500)

# Create vertically concatenated chart with all sections
chart = alt.vconcat(
    make_chart(leaders, "Student leaders"),
    make_chart(gender, "Gender"),
    make_chart(assn, "Associations"),
    make_chart(councils, "Councils"),
    make_chart(cohorts, "Cohorts"),
    make_chart(ticket, "Ticket position", show_xaxis=True),  # Only show x-axis on bottom chart
    spacing=14
).resolve_scale(
    x="shared"  # Share x-axis scale across all charts
).configure_view(
    stroke=None  # Remove chart borders
).configure_axis(
    # Set font for axis labels and titles
    labelFont="Times New Roman",
    titleFont="Times New Roman"
).configure_text(
    # Set font for all text elements
    font="Times New Roman"
)

chart

# Uncomment to save chart as high-resolution PNG
#chart.save("entrants_bar.png", scale_factor=3)

alt.VConcatChart(...)

### **Gender Proportion in Sample Over Time** 

In [8]:
# Filter data to include only students who ran for council
leaders = df[df["ran_for_council"] == 1].copy()

# Group by decade and gender, then count occurrences
agg = (
    leaders
    .groupby(["student_year_decade", "gender"])
    .size()
    .reset_index(name="count")
)

# Calculate proportion of each gender within each decade
agg["prop"] = agg["count"] / agg.groupby("student_year_decade")["count"].transform("sum")

# Create readable gender labels for visualization
agg["gender_label"] = agg["gender"].map({"m": "Male", "f": "Female"})

# Define chronological order for decades
order = ["1930s", "1940s", "1950s", "1960s", "1970s", "1980s", "1990s", "2000s", "2010s", "2020s"]
# Convert decade column to ordered categorical for proper sorting
agg["student_year_decade"] = pd.Categorical(
    agg["student_year_decade"],
    categories=order,
    ordered=True
)

# Create line chart showing gender proportions over time
chart = alt.Chart(agg).mark_line(point=True).encode(
    x=alt.X(
        "student_year_decade:N",
        sort=order,
        title=None,
        axis=alt.Axis(labelAngle=0)
    ),
    y=alt.Y(
        "prop:Q",
        title=None,
        axis=alt.Axis(format="%")
    ),
    color=alt.Color(
        "gender_label:N",
        scale=alt.Scale(domain=["Male", "Female"], range=["#4a4a4a", "#9e9e9e"]),
        legend=alt.Legend(title=None)
    )
).properties(
    width=500
).configure_view(
    stroke=None
).configure_axis(
    labelFont="Times New Roman",
    titleFont="Times New Roman"
).configure_text(
    font="Times New Roman"
).configure_legend(
    labelFont="Times New Roman",
    titleFont="Times New Roman"
)

chart

#chart.save("gender_proportion_line.png", scale_factor=3)

alt.Chart(...)

### **Temporal Changes in Student Association Representation**

In [6]:
def make_stacked_chart(agg, group_col, group_order, header_label, show_x_axis=True, show_legend=True):

    # Create the main stacked bar chart
    bars = alt.Chart(agg).mark_bar(size=18).encode(
        # Set y-axis to group column with specified order, no title
        y=alt.Y(f"{group_col}:N", sort=group_order, title=None),
        # Set x-axis to proportions with normalized stacking
        x=alt.X(
            "prop:Q",
            stack="normalize",
            title=None,
            # Show percentage format if x-axis is visible, otherwise hide all axis elements
            axis=alt.Axis(format="%") if show_x_axis else alt.Axis(labels=False, ticks=False, domain=False)
        ),
        # Color bars by student association with custom color scheme
        color=alt.Color(
            "student_association:N",
            sort=assn_order,
            scale=alt.Scale(domain=assn_order, range=colors),
            # Show legend only if specified
            legend=alt.Legend(title=None) if show_legend else None
        ),
        # Order bars by association order in ascending sequence
        order=alt.Order("assn_order:Q", sort="ascending")
    ).properties(width=500)

    # Create header label data for the chart
    header_df = pd.DataFrame({
        group_col: [group_order[0]],  # Use first group for positioning
        "x0": [0],                    # Position at x=0
        "label": [header_label]       # Header text to display
    })

    # Create header text element with custom styling
    header = alt.Chart(header_df).mark_text(
        align="left",        # Left-align text
        baseline="bottom",   # Align to bottom baseline
        dy=-9,              # Offset text upward by 9 pixels
        fontWeight="bold",   # Make text bold
        fontSize=11         # Set font size
    ).encode(
        y=f"{group_col}:N",  # Position on y-axis
        x="x0:Q",           # Position on x-axis
        text="label:N"      # Text content to display
    )

    # Combine header and bars into single chart
    return header + bars


# Create aggregated data for cohort and decade groupings
agg_cohort = make_agg(leaders, "cohort", cohort_order)
agg_decade = make_agg(leaders, "student_year_decade", decade_order)

# Create cohort chart (no x-axis, with legend)
cohort_chart = make_stacked_chart(
    agg_cohort,
    group_col="cohort",
    group_order=cohort_order,
    header_label="Cohort",
    show_x_axis=False,  # Hide x-axis for top chart
    show_legend=True    # Show legend on top chart
)

# Create decade chart (with x-axis, no legend)
decade_chart = make_stacked_chart(
    agg_decade,
    group_col="student_year_decade",
    group_order=decade_order,
    header_label="Decade",
    show_x_axis=True,   # Show x-axis for bottom chart
    show_legend=False   # Hide legend on bottom chart
)

# Combine charts vertically with custom styling
chart = alt.vconcat(
    cohort_chart,
    decade_chart,
    spacing=14          # Add spacing between charts
).configure_view(
    stroke=None         # Remove chart border
).configure_axis(
    labelFont="Times New Roman",  # Set axis label font
    titleFont="Times New Roman"   # Set axis title font
).configure_text(
    font="Times New Roman"        # Set text font
).configure_legend(
    labelFont="Times New Roman",  # Set legend label font
    titleFont="Times New Roman"   # Set legend title font
)

# Display the final chart
chart

# Uncomment to save chart as high-resolution PNG
# chart.save("association_stacked_bar.png", scale_factor=3)

alt.VConcatChart(...)